In [ ]:
from __future__ import annotations

import os
from typing import Any

import ollama

from agents.agent_models.agent import PromptAgent
from agents.agent_models.generative_lm import GenLM
from agents.agent_models.tools import AgentToolsDisk

In [ ]:
main_dir = os.path.dirname(os.getcwd())

hdf5_path = os.path.join(main_dir, "data/input_data/en_wiki/en_wiki_forward_links.hdf5")
matrix_info_path = os.path.join(
    main_dir, "data/input_data/en_wiki/en_wiki_matrix_index_info.tsv"
)
summary_path = os.path.join(main_dir, "data/input_data/en_wiki/en_wiki_summary.tsv")

In [ ]:
class OllamaLM(GenLM):
    def __init__(self, model: str, options: dict | None = None, keep_alive: int = 60):
        if options is None:
            options = {"temperature": 0.1}
        self.options = options

        self.keep_alive = keep_alive

        if model not in [mod["name"] for mod in ollama.list()["models"]]:
            print("Model not found. Trying to download...")
            ollama.pull(model)

        self.model = model

    def get_response(self, messages: list[dict[Any, Any]]) -> str | None:
        responce = ollama.chat(
            self.model,
            messages=messages,
            format="json",
            options=self.options,
            keep_alive=self.keep_alive,
        )["message"]["content"]

        print(sum([len(m["content"]) for m in messages]), "charaters")
        [print(m["content"]) for m in messages]
        print(responce)
        return responce

In [ ]:
ollama_agent = PromptAgent(
    AgentToolsDisk(
        hdf5_path,
        matrix_info_path,
        open(summary_path, "rb"),  # noqa: SIM115
    ),
    50,
    OllamaLM("llama3.2:latest"),
)

In [ ]:
ollama_agent.play_game(436546, 85743)